## Домашнее задание 3

### Вы — начинающий аналитик / разработчик, который хочет помочь другу найти хорошую вакансию Python-разработчика на hh.ru.

Используя публичный API HeadHunter (базовый URL: https://api.hh.ru), напишите скрипт, который:

* Ищет вакансии по запросу «Python разработчик» в регионе Москва (area=1) с опытом от 1 до 3 лет (experience=between1And3).
* Собирает не менее 200 вакансий, переходя по страницам (page, per_page) _(если столько нет - то все вакансии)_.

* Фильтрует вакансии, оставляя только те, у которых:

    * указана зарплата (salary не null),

    * валюта — рубли (RUR),валюта — рубли (RUR),

    * средняя зарплата не ниже 180 000 ₽.

Средняя зарплата считается так:

* если заданы и from, и to → среднее арифметическое;

* если задано только from → берем from;

* если только to → берем to.

Для каждой оставшейся вакансии скрипт считает индекс привлекательности:

```score = log10(average_salary) + 0.3 * k```


где
k — количество ключевых слов из списка
['Django', 'Flask', 'asyncio', 'pandas', 'SQL'],
которые встречаются в тексте описания вакансии
(поля snippet.requirement и snippet.responsibility, без учета регистра).

Сортирует вакансии по score по убыванию и выводит топ-10 в виде «таблицы» в консоли с колонками:

* название вакансии;
* название компании;
* средняя зарплата;
* score;
* ссылка на вакансию (alternate_url).

В конце выводит среднюю зарплату среди этих 10 лучших вакансий.

In [1]:
import requests
import math
import pandas as pd

In [2]:
# 1. Ищем все вакансии, соответствующие запросу: «Python разработчик в регионе Москва с опытом от 1 до 3 лет".
BASE_URL = "https://api.hh.ru"
VACANCIES_URL = f"{BASE_URL}/vacancies"

params = {
    "text": "Python разработчик",
    "area": 1,  # Москва
    "experience": "between1And3",
    "page": 0,
    "per_page": 100,
}

resp = requests.get(VACANCIES_URL, params=params, timeout=30)
resp.raise_for_status()

data = resp.json()
items = data["items"]

print("Найдено вакансий:", data["found"])
print("Количество на странице:", len(items))
print("\nПервые 5 вакансий:")
for v in items[:5]:
    print("-", v["name"], "|", v["alternate_url"])


Найдено вакансий: 394
Количество на странице: 100

Первые 5 вакансий:
- Разработчик Python | https://hh.ru/vacancy/127880956
- Разработчик JavaScript (frontend) | https://hh.ru/vacancy/128145856
- Python-разработчик | https://hh.ru/vacancy/128247003
- Frontend-разработчик | https://hh.ru/vacancy/127869919
- Тестировщик | https://hh.ru/vacancy/128090577


2. Следующим пунктом оставим 200 первых вакансий (или все, если их меньше).

In [3]:
# 2. Берем первые 200 вакансий

params["page"] = 0

vacancies = []

while True:
    resp = requests.get(VACANCIES_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    vacancies.extend(data["items"])

    # если уже набрали 200 и более - можно остановиться
    if len(vacancies) >= 200:
        vacancies = vacancies[:200]  # оставим ровно 200
        break

    # если страниц больше нет - остановка
    if params["page"] >= data["pages"] - 1:
        break

    params["page"] += 1

print("Отобрано вакансий:", len(vacancies))

Отобрано вакансий: 200


Итак, у нас осталось только 200 вакансий. Оставим из них только те, у кого указана зарплата и она - в рублях. Выведем эту информацию для наглядности и посчитаем среднюю (проверим сразу, что средняя посчиталась по предложенному в задании алгоритму).

In [4]:
# 3. Фильтруем из этих 200 только те, где salary есть и валюта RUR,
# и сразу считаем среднюю зарплату

def avg_salary(sal):
    sal_from = sal.get("from")
    sal_to = sal.get("to")
    if sal_from is not None and sal_to is not None:
        return (sal_from + sal_to) / 2
    elif sal_from is not None:
        return sal_from
    elif sal_to is not None:
        return sal_to
    else:
        return None

vacancies_rur = []
for v in vacancies:
    sal = v.get("salary")
    if sal is not None and sal.get("currency") == "RUR":
        vacancies_rur.append(v)

print("С указанной зарплатой в рублях (из 200):", len(vacancies_rur))

# Вывод: название + зарплата от/до + средняя + ссылка
for v in vacancies_rur:
    sal = v["salary"]
    sal_from = sal.get("from")
    sal_to = sal.get("to")
    cur = sal.get("currency")

    if sal_from is not None and sal_to is not None:
        sal_str = f"от {sal_from} до {sal_to} {cur}"
    elif sal_from is not None:
        sal_str = f"от {sal_from} {cur}"
    elif sal_to is not None:
        sal_str = f"до {sal_to} {cur}"
    else:
        sal_str = "зарплата указана, но без от/до"

    avg = avg_salary(sal)
    avg_str = f"средняя: {avg}" if avg is not None else "средняя: нет данных"

    print(f'{v.get("name", "")} | {sal_str} | {avg_str} | {v.get("alternate_url", "")}')


С указанной зарплатой в рублях (из 200): 63
Python-разработчик | от 150000 до 200000 RUR | средняя: 175000.0 | https://hh.ru/vacancy/128247003
Frontend-разработчик | от 250000 RUR | средняя: 250000 | https://hh.ru/vacancy/127869919
Тестировщик | до 100000 RUR | средняя: 100000 | https://hh.ru/vacancy/128090577
Backend-разработчик | от 200000 RUR | средняя: 200000 | https://hh.ru/vacancy/128087946
Разработчик программного обеспечения (искусственный интеллект) | от 100000 до 120000 RUR | средняя: 110000.0 | https://hh.ru/vacancy/128290735
Data Engineer / Data Analyst / аналитик данных (middle) | от 100000 до 200000 RUR | средняя: 150000.0 | https://hh.ru/vacancy/128325537
Python разработчик (LLM, FastAPI, микросервисы) | до 300000 RUR | средняя: 300000 | https://hh.ru/vacancy/127254705
Python-разработчик | от 60000 до 100000 RUR | средняя: 80000.0 | https://hh.ru/vacancy/127433763
Middle QA-инженер / Тестировщик | от 110000 RUR | средняя: 110000 | https://hh.ru/vacancy/127718739
Junior b

Проверили, что средняя заработная плата считается корректно, теперь оставим только те вакансии, у которых она более 180 000 руб.

In [6]:
# 4. Оставляем только вакансии со средней зарплатой >= 180000 руб.
vacancies_180 = []
for v in vacancies_rur:
    avg = avg_salary(v["salary"])  # считаем среднюю зп
    if avg is not None and avg >= 180000:
        vacancies_180.append((v, avg))  # сохраняем avg вместе с вакансией

print("Отобрано вакансий всего:", len(vacancies))
print("Из них с указанной зарплатой в рублях:", len(vacancies_rur))
print("Из них средняя зарплата >= 180 000 руб.:", len(vacancies_180))

# вывод
for v, avg in vacancies_180:
    sal = v["salary"]
    sal_from = sal.get("from")
    sal_to = sal.get("to")
    cur = sal.get("currency")

    if sal_from is not None and sal_to is not None:
        sal_str = f"от {sal_from} до {sal_to} {cur}"
    elif sal_from is not None:
        sal_str = f"от {sal_from} {cur}"
    elif sal_to is not None:
        sal_str = f"до {sal_to} {cur}"
    else:
        sal_str = "зарплата указана, но без от/до"

    print(f'{v.get("name", "")} | {sal_str} | средняя: {avg} | {v.get("alternate_url", "")}')


Отобрано вакансий всего: 200
Из них с указанной зарплатой в рублях: 63
Из них средняя зарплата >= 180 000 руб.: 21
Frontend-разработчик | от 250000 RUR | средняя: 250000 | https://hh.ru/vacancy/127869919
Backend-разработчик | от 200000 RUR | средняя: 200000 | https://hh.ru/vacancy/128087946
Python разработчик (LLM, FastAPI, микросервисы) | до 300000 RUR | средняя: 300000 | https://hh.ru/vacancy/127254705
Ведущий разработчик искусственного интеллекта/Middle Data scientist | от 180000 до 180000 RUR | средняя: 180000.0 | https://hh.ru/vacancy/128307105
Prompt-инженер | до 180000 RUR | средняя: 180000 | https://hh.ru/vacancy/127577430
Инженер технической поддержки | до 270000 RUR | средняя: 270000 | https://hh.ru/vacancy/127787550
Личный ассистент технологического предпринимателя | от 150000 до 300000 RUR | средняя: 225000.0 | https://hh.ru/vacancy/127943836
Разработчик/инженер машинного обучения / ML Engineer | до 250000 RUR | средняя: 250000 | https://hh.ru/vacancy/127379557
DWH Develope

Теперь посчитаем количество ключевых слов, индекс привлекательности ( отсортируем нашу выборку по убыванию по данному признаку). Оставим только среднюю зарплату - данные о минимуме и максимуме заработка больше не нужны.

In [7]:
# 5. Считаем количество ключевых слов, индекс привлекательности. Данные выводим в виде "таблицы" в консоли

keywords = ['Django', 'Flask', 'asyncio', 'pandas', 'SQL']

def keyword_count(vac):
    snip = vac.get("snippet") or {}
    text = ((snip.get("requirement") or "") + " " + (snip.get("responsibility") or "")).lower()
    return sum(1 for kw in keywords if kw.lower() in text)

rows = []
for v, avg in vacancies_180:   # <-- важно: распаковка (v, avg)
    k = keyword_count(v)
    score = math.log10(avg) + 0.3 * k

    rows.append({
        "Название вакансии": v.get("name", ""),
        "Средняя заработная плата": avg,
        "Количество ключевых слов": k,
        "Индекс привлекательности": score,
        "Ссылка на вакансию": v.get("alternate_url", "")
    })

scores_df = pd.DataFrame(rows).sort_values("Индекс привлекательности", ascending=False)
scores_df


,Название вакансии,Средняя заработная плата,Количество ключевых слов,Индекс привлекательности,Ссылка на вакансию
8,DWH Developer/Разработчик DWH,400000.0,1,5.902060,https://hh.ru/vacancy/127786460
9,Data Analyst / Аналитик данных (ETL & Data Col...,200000.0,2,5.901030,https://hh.ru/vacancy/127804244
2,"Python разработчик (LLM, FastAPI, микросервисы)",300000.0,1,5.777121,https://hh.ru/vacancy/127254705
5,Инженер технической поддержки,270000.0,1,5.731364,https://hh.ru/vacancy/127787550
20,Product Analyst / Marketing Analyst,260000.0,1,5.714973,https://hh.ru/vacancy/127822917
15,Специалист по предиктивной аналитике,250000.0,1,5.697940,https://hh.ru/vacancy/128008125
0,Frontend-разработчик,250000.0,1,5.697940,https://hh.ru/vacancy/127869919
18,Разработчик Python,243500.0,1,5.686499,https://hh.ru/vacancy/87838855
1,Backend-разработчик,200000.0,1,5.601030,https://hh.ru/vacancy/128087946
11,Инженер технической поддержки и внедрения ПО,200000.0,1,5.601030,https://hh.ru/vacancy/127708968


Теперь из получившегося списка отберем топ-10 вакансий по индексу привлекательности и добавим названиее компании. В конце посчитаем среднюю зарплату по полученной выборке.

In [8]:
# 6. Топ-10 по индексу привлекательности + название компании + средняя зарплата по топ-10

url_to_company = {
    v.get("alternate_url", ""): (v.get("employer") or {}).get("name", "")
    for v, avg in vacancies_180
}

scores_df["Название компании"] = scores_df["Ссылка на вакансию"].map(url_to_company)

top10 = scores_df.head(10).copy()
top10.insert(0, "№", range(1, len(top10) + 1))  # нумерация 1..10

print(top10[[
    "№",
    "Название вакансии",
    "Название компании",
    "Средняя заработная плата",
    "Индекс привлекательности",
    "Ссылка на вакансию"
]].to_string(index=False))

print("\nСредняя зарплата среди топ-10:",
      f'{round(top10["Средняя заработная плата"].mean(), 2)} руб.')


 №                                      Название вакансии            Название компании  Средняя заработная плата  Индекс привлекательности              Ссылка на вакансию
 1                          DWH Developer/Разработчик DWH                 Топассистент                  400000.0                  5.902060 https://hh.ru/vacancy/127786460
 2 Data Analyst / Аналитик данных (ETL & Data Collection)     Уфимцева Ольга Сергеевна                  200000.0                  5.901030 https://hh.ru/vacancy/127804244
 3        Python разработчик (LLM, FastAPI, микросервисы) Меньщиков Павел Владимирович                  300000.0                  5.777121 https://hh.ru/vacancy/127254705
 4                          Инженер технической поддержки                       Латера                  270000.0                  5.731364 https://hh.ru/vacancy/127787550
 5                    Product Analyst / Marketing Analyst                       ROSSKO                  260000.0                  5.714973 https:

А теперь из указанных выше 6 шагов соберем единый скрипт без лишних выводов промежуточных данных:

In [9]:
import requests
import math
import pandas as pd

BASE_URL = "https://api.hh.ru"
VACANCIES_URL = f"{BASE_URL}/vacancies"

KEYWORDS = ['Django', 'Flask', 'asyncio', 'pandas', 'SQL']
MIN_AVG = 180000

params = {
    "text": "Python разработчик",
    "area": 1,                      # Москва
    "experience": "between1And3",    # 1-3 года
    "page": 0,
    "per_page": 100,
}

def avg_salary(sal: dict):
    """Расчет средней заработной платы. Возвращает число или None."""
    if not sal:
        return None
    sal_from = sal.get("from")
    sal_to = sal.get("to")
    if sal_from is not None and sal_to is not None:
        return (sal_from + sal_to) / 2
    if sal_from is not None:
        return sal_from
    if sal_to is not None:
        return sal_to
    return None


def keyword_count(vac: dict):
    """Считает, сколько ключевых слов встречаются хотя бы 1 раз (без учёта регистра)."""
    snip = vac.get("snippet") or {}
    text = ((snip.get("requirement") or "") + " " + (snip.get("responsibility") or "")).lower()
    return sum(1 for kw in KEYWORDS if kw.lower() in text)


# 1) Собираем не менее 200 вакансий (если меньше - все)
params["page"] = 0
vacancies = []

while True:
    resp = requests.get(VACANCIES_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    vacancies.extend(data["items"])

    if len(vacancies) >= 200:
        vacancies = vacancies[:200]
        break

    if params["page"] >= data["pages"] - 1:
        break

    params["page"] += 1


# 2) Фильтруем: salary!=None, currency==RUR, avg>=180000; считаем score
rows = []

for v in vacancies:
    sal = v.get("salary")
    if sal is None or sal.get("currency") != "RUR":
        continue

    avg = avg_salary(sal)
    if avg is None or avg < MIN_AVG:
        continue

    k = keyword_count(v)
    score = math.log10(avg) + 0.3 * k

    rows.append({
        "Название вакансии": v.get("name", ""),
        "Название компании": (v.get("employer") or {}).get("name", ""),
        "Средняя зарплата": avg,
        "Индекс привлекательности": score,
        "Ссылка на вакансию": v.get("alternate_url", ""),
    })

df = pd.DataFrame(rows).sort_values("Индекс привлекательности", ascending=False)

top10 = df.head(10).copy()
top10.insert(0, "№", range(1, len(top10) + 1))

print(top10[["№", "Название вакансии", "Название компании", "Средняя зарплата", "Индекс привлекательности", "Ссылка на вакансию"]]
      .to_string(index=False))

if len(top10) > 0:
    print("\nСредняя зарплата среди топ-10:", f"{round(top10['Средняя зарплата'].mean(), 2)} руб.")
else:
    print("\nТоп-10 не сформирован: после фильтрации не осталось вакансий.")


 №                                      Название вакансии            Название компании  Средняя зарплата  Индекс привлекательности              Ссылка на вакансию
 1                          DWH Developer/Разработчик DWH                 Топассистент          400000.0                  5.902060 https://hh.ru/vacancy/127786460
 2 Data Analyst / Аналитик данных (ETL & Data Collection)     Уфимцева Ольга Сергеевна          200000.0                  5.901030 https://hh.ru/vacancy/127804244
 3        Python разработчик (LLM, FastAPI, микросервисы) Меньщиков Павел Владимирович          300000.0                  5.777121 https://hh.ru/vacancy/127254705
 4                          Инженер технической поддержки                       Латера          270000.0                  5.731364 https://hh.ru/vacancy/127787550
 5                    Product Analyst / Marketing Analyst                       ROSSKO          260000.0                  5.714973 https://hh.ru/vacancy/127822917
 6                   С